# Day 4 — Series and DataFrame: schema, no index

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** on Medium (link added after publishing)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [ ]:
import polars as pl

print("polars", pl.__version__)

Two building blocks: a **Series** is one typed column, a **DataFrame** is a set of equal-length Series with names.
What is *missing* matters as much: there is no index.

## 1. Series: one name, one dtype

In [ ]:
prices = pl.Series("price", [58.90, 239.90, 199.00])
print(prices)
print(prices.dtype, prices.len())

In [ ]:
# Types are strict: mixed types do not silently become 'object'
try:
    pl.Series("mixed", [1, "two", 3.0])
except Exception as err:
    print(type(err).__name__, "->", err)

print(pl.Series("mixed", [1, "two", 3.0], strict=False))  # opt in to a common type

In [ ]:
# Missing values are null, for every dtype
pl.Series("installments", [1, None, 3], dtype=pl.Int16)

## 2. DataFrame: named Series of equal length

In [ ]:
df = pl.DataFrame(
    {
        "order_id": ["a1", "b2", "c3"],
        "status": ["delivered", "shipped", "delivered"],
        "price": [58.90, 239.90, 199.00],
    }
)
df

In [ ]:
print(df.schema)   # name -> dtype, the contract of the frame
print(df.columns)
print(df.dtypes)
print(df.shape)

In [ ]:
# Declare the schema up front when you know it
pl.DataFrame(
    {"order_id": ["a1"], "installments": [3]},
    schema={"order_id": pl.String, "installments": pl.UInt8},
)

## 3. No index — and why that is a feature

In pandas, arithmetic between two Series **aligns on the index**. That is powerful, and also a classic source of silent NaNs.

In [ ]:
import pandas as pd

a = pd.Series([10, 20, 30], index=[0, 1, 2])
b = pd.Series([1, 2, 3], index=[1, 2, 3])
print(a + b)  # aligned on index labels -> NaN at 0 and 3

In [ ]:
# Polars works by position: same length in, same length out
pl.Series([10, 20, 30]) + pl.Series([1, 2, 3])

Need a row number? Make it an explicit column. Need a lookup by key? Use `filter` or a `join`.

In [ ]:
orders = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)

orders.with_row_index("row_nr").head(3)

In [ ]:
# "loc by key" becomes a filter
orders.filter(pl.col("order_id") == "e481f51cbdc54678b7cc49136f2d6af7")

## 4. First look at a real table

In [ ]:
orders.glimpse()

In [ ]:
orders.describe()

In [ ]:
# Getting at values
print(orders["order_status"].head(3))   # a Series
print(orders.row(0))                     # one row as a tuple
print(orders.select(pl.len()).item())    # a single scalar

## 5. Moving between pandas and Polars

In [ ]:
orders_pd = orders.to_pandas()
print(type(orders_pd), orders_pd.shape)

back = pl.from_pandas(orders_pd)
print(back.shape == orders.shape)
back.schema  # compare with orders.schema: check the datetime precision after the round trip

## Try it yourself

1. Build a DataFrame with a column of `None` only. What dtype does Polars pick? How do you set it?
2. In pandas, `orders_pd.set_index("order_id")` and look up one order with `.loc`. Write the Polars equivalent without an index.
3. What does `orders.schema` say about `order_purchase_timestamp` with and without `try_parse_dates=True`?